# Stage 2: Coding the Attention mechanism

Before jumping into code a full multi-head attention, we'll go step by step.

1. Embeddings → QxKt →
2. Attention Scores → 
3. Scale → 
4. Causal Mask → 
5. Softmax → 
6. Attention Weights → ×V → 
7. Context

## 1. Simple Self-attention

Without parameters

In [1]:
import torch

inputs = torch.tensor([
    [0.43, 0.15, 0.89],  # Your
    [0.55, 0.87, 0.66],  # journey
    [0.57, 0.85, 0.64],  # starts
    [0.22, 0.58, 0.33],  # with
    [0.77, 0.25, 0.10],  # one
    [0.05, 0.80, 0.55],  # step
])

print(inputs.shape)
# (T, C) = (6, 3)

torch.Size([6, 3])


In [2]:
query = inputs[1]

In [3]:
attention_scores = torch.empty(inputs.shape[0])

for i, token_embedding in enumerate(inputs):
    attention_scores[i] = torch.dot(
        query,
        token_embedding,
    )

print(attention_scores)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


Now, we apply softmax to normalize. It's better than regular normalization as it handles well negative input an sum to zero thanks to Euler number

In [4]:
attention_weights = torch.softmax(
    attention_scores,
    dim=0
)

print(attention_weights)
print(attention_weights.sum())

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(1.)


Now, obtain the context vector

In [ ]:
context_vector = torch.zeros(query.shape)

for i, token_embedding in enumerate(inputs):
    context_vector += (
        attention_weights[i] * token_embedding
    )

print(context_vector)
print(context_vector.shape)

tensor([0.4419, 0.6515, 0.5683])
torch.Size([3])


Now, we'll obtain the context vector for each input token

In [6]:
attention_scores = inputs @ inputs.T

print(attention_scores)
print(attention_scores.shape)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
torch.Size([6, 6])


In [12]:
attention_weights = torch.softmax(
    attention_scores,
    dim=-1
)

print(attention_weights)
print(attention_weights.shape)
print(attention_weights.sum(dim=-1))

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
torch.Size([6, 6])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [15]:
context_vectors = attention_weights @ inputs

print(context_vectors)
print(context_vectors.shape)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
torch.Size([6, 3])


## 2. Add trainable parameters Q, K, V

In [ ]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [19]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(
    torch.rand(d_in, d_out),
    requires_grad=False
)

W_key = torch.nn.Parameter(
    torch.rand(d_in, d_out),
    requires_grad=False
)

W_value = torch.nn.Parameter(
    torch.rand(d_in, d_out),
    requires_grad=False
)

print(W_query)

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])


In [20]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print("Original:", x_2, x_2.shape)
print("Query:", query_2, query_2.shape)
print("Key:", key_2, key_2.shape)
print("Value:", value_2, value_2.shape)

Original: tensor([0.5500, 0.8700, 0.6600]) torch.Size([3])
Query: tensor([0.4306, 1.4551]) torch.Size([2])
Key: tensor([0.4433, 1.1419]) torch.Size([2])
Value: tensor([0.3951, 1.0037]) torch.Size([2])


Now we transform all input tokens

In [21]:
queries = inputs @ W_query
keys = inputs @ W_key
values = inputs @ W_value

print("Queries:", queries.shape)
print("Keys:", keys.shape)
print("Values:", values.shape)

Queries: torch.Size([6, 2])
Keys: torch.Size([6, 2])
Values: torch.Size([6, 2])


Query: what the token wants to know

Key: how the token can be found

Value: what information does the token provide∫

In [ ]:
attention_scores_2 = query_2 @ keys.T
# how much the token 2 must look at other tokens

print(attention_scores_2)
print(attention_scores_2.shape)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])
torch.Size([6])


Now, we need to scale the scores and apply softmax to obtain attention weights

attention weights = softmax(QKᵀ / √dₖ)

In [29]:
import math

attention_weights_2 = torch.softmax(
    attention_scores_2 / math.sqrt(d_out),
    dim = -1
)

print(attention_weights_2)
print(attention_weights_2.sum())

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
tensor(1.)


We need to scale because with larger scores, softmax tends to saturate

Now that we have how much each token looks at each other, we need to take the information they can provide --> CONTEXT VECTOR

In [34]:
context_vector_2 = attention_weights_2 @ values

print(context_vector_2)
print(context_vector_2.shape)

tensor([0.3061, 0.8210])
torch.Size([2])


At this point, we already implemented how query, key and value work together. The next step will be to create an standard function for implementing Self Attention.

In [37]:
import torch
import torch.nn as nn

class SelfAttentionV1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()

        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        queries = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value

        att_scores = queries @ keys.T

        att_weights = torch.softmax(
            att_scores / keys.shape[-1] ** 0.5,
            dim = -1
        )

        context_vectors = att_weights @ values

        return context_vectors

In [38]:
torch.manual_seed(123)

self_attention = SelfAttentionV1(
    d_in=inputs.shape[1],
    d_out=2,
)

context_vectors = self_attention(inputs)

print(context_vectors)
print(context_vectors.shape)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)
torch.Size([6, 2])


Before jumping into the next attention mechanism, it's more common in PyTorch to use nn.Linear instead of nn.Parameter

In [ ]:
# with nn.Linear

class SelfAttentionV2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attention_scores = queries @ keys.T

        attention_weights = torch.softmax(
            attention_scores / keys.shape[-1] ** 0.5,
            dim = -1
        )

        context_vectors = attention_weights @ values

        return context_vectors

In [ ]:
torch.manual_seed(789)

self_attention = SelfAttentionV2(
    d_in=inputs.shape[1],
    d_out=2,
)

context_vectors = self_attention(inputs)

print(context_vectors)
print(context_vectors.shape)

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)
torch.Size([6, 2])


## 3. Add Causal Attention (masked)

In reality, a token can't look at the future when it generates the next one, as it is autoregressive. It needs to look at the past to predict the future. For this reason, we need to 'mask' the future tokens.

In [42]:
print(attention_weights)
print(attention_weights.sum(dim=-1))

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


For the softmax, it takes into account that -inf is weight = 0, as it works with the number e

In [46]:
queries = self_attention.W_query(inputs)
keys = self_attention.W_key(inputs)
values = self_attention.W_value(inputs)

attention_scores = queries @ keys.T

sequence_length = inputs.shape[0]

causal_mask = torch.triu(
    torch.ones(
        sequence_length,
        sequence_length,
        device=inputs.device,
        dtype=torch.bool,
    ),
    diagonal=1,
)

masked_scores = attention_scores.masked_fill(
    causal_mask,
    -torch.inf,
)

attention_weights = torch.softmax(
    masked_scores / keys.shape[-1] ** 0.5,
    dim=-1,
)

context_vectors = attention_weights @ values

print(attention_weights)
print(context_vectors)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)
tensor([[-0.0872,  0.0286],
        [-0.0991,  0.0501],
        [-0.0999,  0.0633],
        [-0.0983,  0.0489],
        [-0.0514,  0.1098],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


Before finishing the Causal Mask stage, we need to implement dropout to prevent overfitting.

In [47]:
torch.manual_seed(123)

dropout = nn.Dropout(p=0.5)

dropped_attention_weights = dropout(attention_weights)

print(dropped_attention_weights)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


In [48]:
context_vectors = dropped_attention_weights @ values

print(context_vectors)
print(context_vectors.shape)

tensor([[-0.1744,  0.0572],
        [ 0.0000,  0.0000],
        [-0.1999,  0.1267],
        [-0.1061,  0.0833],
        [-0.0795,  0.0294],
        [-0.0534,  0.1748]], grad_fn=<MmBackward0>)
torch.Size([6, 2])


Now, we're creating a standarized class for Causal Attention

In [52]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()

        self.d_out = d_out

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.dropout = nn.Dropout(dropout)

        self.causal_mask: torch.Tensor

        self.register_buffer(
            "causal_mask",
            torch.triu(
                torch.ones(
                    context_length,
                    context_length,
                    dtype=torch.bool,
                ),
                diagonal=1,
            ),
        )

    def forward(self, x):
        batch_size, num_tokens, d_in = x.shape

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attention_scores = queries @ keys.transpose(1, 2)

        active_mask = self.causal_mask[
            :num_tokens,
            :num_tokens,
        ]

        attention_scores.masked_fill_(
            active_mask,
            -torch.inf,
        )

        attention_weights = torch.softmax(
            attention_scores / keys.shape[-1] ** 0.5,
            dim=-1,
        )

        attention_weights = self.dropout(
            attention_weights
        )

        context_vectors = attention_weights @ values

        return context_vectors

In [ ]:
batch = torch.stack((inputs, inputs), dim=0)

causal_attention = CausalAttention(
    d_in=3,
    d_out=2,
    context_length=6,
    dropout=0.0,
)

context_vectors = causal_attention(batch)

print(context_vectors)
print(context_vectors.shape)

tensor([[[-0.4821,  0.4336],
         [-0.5368,  0.5483],
         [-0.5545,  0.5886],
         [-0.4937,  0.5311],
         [-0.4589,  0.5169],
         [-0.4479,  0.4971]],

        [[-0.4821,  0.4336],
         [-0.5368,  0.5483],
         [-0.5545,  0.5886],
         [-0.4937,  0.5311],
         [-0.4589,  0.5169],
         [-0.4479,  0.4971]]], grad_fn=<UnsafeViewBackward0>)
torch.Size([2, 6, 2])


# 4. Multi Head Attention

Single Head attetion does not capture all the relationship within the text.

So, we're about to implement a new multihead attention mechanism. We're each head represent d_out / num_heads features and then concatenate them.

In [54]:
class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        d_in: int,
        d_out: int,
        context_length: int,
        dropout: float,
        num_heads: int,
        qkv_bias: bool = False,
    ):
        super().__init__()

        assert d_out % num_heads == 0

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias,
        )
        self.W_key = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias,
        )
        self.W_value = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias,
        )

        self.out_proj = nn.Linear(
            d_out,
            d_out,
        )

        self.dropout = nn.Dropout(dropout)

        self.causal_mask: torch.Tensor

        self.register_buffer(
            "causal_mask",
            torch.triu(
                torch.ones(
                    context_length,
                    context_length,
                    dtype=torch.bool,
                ),
                diagonal=1,
            ),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, num_tokens, _ = x.shape

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        queries = queries.view(
            batch_size,
            num_tokens,
            self.num_heads,
            self.head_dim,
        )

        keys = keys.view(
            batch_size,
            num_tokens,
            self.num_heads,
            self.head_dim,
        )

        values = values.view(
            batch_size,
            num_tokens,
            self.num_heads,
            self.head_dim,
        )

        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        attention_scores = (
            queries @ keys.transpose(2, 3)
        )

        active_mask = self.causal_mask[
            :num_tokens,
            :num_tokens,
        ]

        attention_scores.masked_fill_(
            active_mask,
            -torch.inf,
        )

        attention_weights = torch.softmax(
            attention_scores
            / self.head_dim**0.5,
            dim=-1,
        )

        attention_weights = self.dropout(
            attention_weights
        )

        context_vectors = attention_weights @ values

        context_vectors = (
            context_vectors
            .transpose(1, 2)
            .contiguous()
            .view(
                batch_size,
                num_tokens,
                self.d_out,
            )
        )

        context_vectors = self.out_proj(
            context_vectors
        )

        return context_vectors

In [56]:
batch = torch.stack(
    (inputs, inputs),
    dim=0,
)

torch.manual_seed(123)

multihead_attention = MultiHeadAttention(
    d_in=3,
    d_out=4,
    context_length=6,
    dropout=0.0,
    num_heads=2,
)

context_vectors = multihead_attention(batch)

print("Input:", batch)
print("Output:", context_vectors)

Input: tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])
Output: tensor([[[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0178,  0.3221, -0.0763, -0.4225],
         [-0.0147,  0.3259, -0.0734, -0.3721],
         [-0.0116,  0.3138, -0.0708, -0.3624],
         [-0.0117,  0.2973, -0.0698, -0.3543],
         [-0.0132,  0.2990, -0.0689, -0.3490]],

        [[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0178,  0.3221, -0.0763, -0.4225],
         [-0.0147,  0.3259, -0.0734, -0.3721],
         [-0.0116,  0.3138, -0.0708, -0.3624],
         [-0.0117,  0.2973, -0.0698, -0.3543],
         [-0.0132,  0.2990, -0.0689, -0.34

## Summary

- Self-attention creates a contextual representation for every token.
- Queries and keys determine attention scores.
- Attention weights combine the values.
- Scaling prevents softmax saturation.
- The causal mask prevents tokens from attending to future positions.
- Dropout regularizes attention during training.
- Multi-head attention performs several independent attention operations.
- Each head works with `head_dim = d_out / num_heads`.
- Head outputs are concatenated and mixed through an output projection.

### Main shapes

- Input: `(B, T, d_in)`
- Q, K, V before splitting: `(B, T, d_out)`
- Q, K, V after splitting: `(B, H, T, head_dim)`
- Attention scores: `(B, H, T, T)`
- Output: `(B, T, d_out)`